In [0]:
# Set spark config

spark.conf.set("spark.sql.shuffle.partitions", 50)

In [0]:
df = spark.read.format("csv").option("header", True).load("/Volumes/new_external_catalog/default/test-vol/Employee_Attrition.csv")
display(df)

In [0]:
# Transformations on dataframe and write data into delta table

from pyspark.sql.functions import col

high_risk_df = df.filter(
    (col("Attrition") == "No") & (col("JobSatisfaction").cast("int") < 3)
)

selected_columns = [
    "EmployeeNumber", "EmployeeName", "Department", "JobRole", "JobSatisfaction",
    "Age", "Gender", "MaritalStatus", "MonthlyIncome", "OverTime", "YearsAtCompany"
]
high_risk_selected_df = high_risk_df.select(*[c for c in selected_columns if c in high_risk_df.columns])

high_risk_selected_df.write.format("delta").mode("overwrite").saveAsTable("`new_external_catalog`.default.high_risk_attrition_employees")

In [0]:
history_df = spark.sql("DESCRIBE HISTORY `new_external_catalog`.default.high_risk_attrition_employees")

display(history_df.select("version", "timestamp", "operation"))

In [0]:

delta_df = spark.read.format("delta").table("`new_external_catalog`.default.high_risk_attrition_employees")
display(delta_df)

In [0]:
from pyspark.sql import Row

dummy_data = [Row(
    EmployeeNumber="999999",
    Department="Dummy Dept",
    JobRole="Dummy Role",
    JobSatisfaction="1",
    Age="30",
    Gender="Other",
    MaritalStatus="Single",
    MonthlyIncome="0",
    OverTime="No",
    YearsAtCompany="0"
)]

dummy_df = spark.createDataFrame(dummy_data)

dummy_df.write.format("delta").mode("append").saveAsTable("`new_external_catalog`.default.high_risk_attrition_employees")

In [0]:
# Fetch all versions of delta table

history_df = spark.sql("DESCRIBE HISTORY `new_external_catalog`.default.high_risk_attrition_employees")
display(history_df.select("version", "timestamp", "operation"))

In [0]:
# Read latest data from delta table

delta_df = spark.read.format("delta").table("`new_external_catalog`.default.high_risk_attrition_employees")
display(delta_df)

In [0]:
# Read data from specific version of delta table

delta_df_v1 = spark.read.format("delta").option("versionAsOf", 0).table("`new_external_catalog`.default.high_risk_attrition_employees")
display(delta_df_v1)

In [0]:

spark.sql("CREATE VOLUME IF NOT EXISTS `new_external_catalog`.default.employee_transformed_data")

In [0]:
# Write partitioned data into Volume path

from pyspark.sql.functions import col, upper

# Example logical transformation: uppercase JobRole for all employees
transformed_df = df.withColumn("JobRole", upper(col("JobRole")))

output_path = "/Volumes/new_external_catalog/default/employee_transformed_data/"

transformed_df.write.mode("overwrite").partitionBy("Department").format("parquet").save(output_path)